In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
from imblearn.combine import SMOTEENN, SMOTETomek
import xgboost as xgb


In [2]:
df=pd.read_csv(r"E:\credit_risk_modeling\data\final_dataset.csv")

In [4]:
X=df.drop('Approved_Flag', axis=1)
y=df['Approved_Flag']

In [5]:
x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [6]:
import numpy as np
from sklearn.utils.class_weight import compute_class_weight

classes = np.unique(y_train)
class_weights = compute_class_weight(class_weight='balanced', classes=classes, y=y_train)

# Convert to dict
class_weight_dict = dict(zip(classes, class_weights))

# Create sample weights
sample_weights = np.array([class_weight_dict[y] for y in y_train])


In [7]:
import xgboost as xgb

xgb_clf = xgb.XGBClassifier(
    objective='multi:softprob',
    num_class=4,
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.5,
    reg_lambda=1.0,
    random_state=42,
    n_jobs=-1
)

xgb_clf.fit(
    x_train,
    y_train,
    sample_weight=sample_weights
)


,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'multi:softprob'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,0.8
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes fr

In [8]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# Predictions
y_pred_train = xgb_clf.predict(x_train)
y_pred_test = xgb_clf.predict(x_test)

print("=== TRAIN REPORT ===")
print(classification_report(y_train, y_pred_train))

print("=== TEST REPORT ===")
print(classification_report(y_test, y_pred_test))

print("Train Accuracy:", accuracy_score(y_train, y_pred_train))
print("Test Accuracy:", accuracy_score(y_test, y_pred_test))

print("Confusion Matrix (Test):")
print(confusion_matrix(y_test, y_pred_test))


=== TRAIN REPORT ===
              precision    recall  f1-score   support

           0       0.49      0.93      0.64      3894
           1       0.86      0.50      0.64     20407
           2       0.38      0.51      0.43      5115
           3       0.40      0.70      0.51      4235

    accuracy                           0.58     33651
   macro avg       0.53      0.66      0.55     33651
weighted avg       0.68      0.58      0.59     33651

=== TEST REPORT ===
              precision    recall  f1-score   support

           0       0.46      0.84      0.59      1014
           1       0.77      0.46      0.58      5045
           2       0.27      0.33      0.30      1325
           3       0.28      0.51      0.36      1029

    accuracy                           0.49      8413
   macro avg       0.44      0.54      0.46      8413
weighted avg       0.59      0.49      0.51      8413

Train Accuracy: 0.5786752251047517
Test Accuracy: 0.4928087483656246
Confusion Matrix (Te

In [13]:
import pandas as pd

importance = xgb_clf.feature_importances_
feat_imp = pd.Series(importance, index=x_train.columns).sort_values(ascending=False)

print(feat_imp.head(30))

Age_Oldest_TL                   0.157965
last_prod_enq2_others           0.072261
last_prod_enq2_ConsumerLoan     0.056316
pct_tl_open_L12M                0.052021
first_prod_enq2_others          0.033709
Home_TL                         0.033674
last_prod_enq2_PL               0.032362
Secured_TL                      0.031841
pct_tl_open_L6M                 0.031167
first_prod_enq2_ConsumerLoan    0.030513
last_prod_enq2_HL               0.030153
Other_TL                        0.029298
first_prod_enq2_PL              0.029236
MARITALSTATUS_Single            0.028519
PL_TL                           0.028055
Age_Newest_TL                   0.027737
pct_active_tl                   0.025066
pct_closed_tl                   0.024956
Unsecured_TL                    0.023283
Total_TL_opened_L12M            0.023111
pct_tl_closed_L12M              0.023040
CC_TL                           0.021817
first_prod_enq2_CC              0.021617
last_prod_enq2_CC               0.021486
pct_tl_closed_L6

In [14]:
from sklearn.metrics import confusion_matrix
print(confusion_matrix(y_test, y_pred_test))


[[ 850  137    8   19]
 [ 863 2328  925  929]
 [ 120  367  442  396]
 [  33  193  277  526]]


In [10]:
new=df.columns.to_list()

In [11]:
new

['pct_tl_open_L6M',
 'pct_tl_closed_L6M',
 'pct_active_tl',
 'pct_closed_tl',
 'Total_TL_opened_L12M',
 'Tot_TL_closed_L12M',
 'pct_tl_open_L12M',
 'pct_tl_closed_L12M',
 'Tot_Missed_Pmnt',
 'CC_TL',
 'Home_TL',
 'PL_TL',
 'Secured_TL',
 'Unsecured_TL',
 'Other_TL',
 'Age_Oldest_TL',
 'Age_Newest_TL',
 'EDUCATION',
 'Approved_Flag',
 'MARITALSTATUS_Single',
 'GENDER_M',
 'last_prod_enq2_CC',
 'last_prod_enq2_ConsumerLoan',
 'last_prod_enq2_HL',
 'last_prod_enq2_PL',
 'last_prod_enq2_others',
 'first_prod_enq2_CC',
 'first_prod_enq2_ConsumerLoan',
 'first_prod_enq2_HL',
 'first_prod_enq2_PL',
 'first_prod_enq2_others']

In [ ]:
# from imblearn.over_sampling import SMOTE

# smote = SMOTE(sampling_strategy=0.5, random_state=42)  
# # only partially balance

# X_train_sm, y_train_sm = smote.fit_resample(x_train, y_train)


In [16]:
import xgboost as xgb
from sklearn.model_selection import GridSearchCV

xgb_clf = xgb.XGBClassifier(
    objective='multi:softprob',
    num_class=4,
    random_state=42,
    n_jobs=-1
)

param_grid = {
    'n_estimators': [200, 300],
    'max_depth': [5, 7],
    'learning_rate': [0.03, 0.05],
    'subsample': [0.8, 0.9],
    'colsample_bytree': [0.8, 0.9],
    'gamma': [0, 1],
    'reg_alpha': [0, 1],
    'reg_lambda': [1, 2]
}

grid_search = GridSearchCV(
    estimator=xgb_clf,
    param_grid=param_grid,
    scoring='f1_macro',
    cv=3,
    verbose=2,
    n_jobs=1
)

grid_search.fit(x_train, y_train, sample_weight=sample_weights)

print("Best Params:", grid_search.best_params_)
print("Best Score:", grid_search.best_score_)


Fitting 3 folds for each of 256 candidates, totalling 768 fits
[CV] END colsample_bytree=0.8, gamma=0, learning_rate=0.03, max_depth=5, n_estimators=200, reg_alpha=0, reg_lambda=1, subsample=0.8; total time=   3.4s
[CV] END colsample_bytree=0.8, gamma=0, learning_rate=0.03, max_depth=5, n_estimators=200, reg_alpha=0, reg_lambda=1, subsample=0.8; total time=   2.4s
[CV] END colsample_bytree=0.8, gamma=0, learning_rate=0.03, max_depth=5, n_estimators=200, reg_alpha=0, reg_lambda=1, subsample=0.8; total time=   3.0s
[CV] END colsample_bytree=0.8, gamma=0, learning_rate=0.03, max_depth=5, n_estimators=200, reg_alpha=0, reg_lambda=1, subsample=0.9; total time=   2.7s
[CV] END colsample_bytree=0.8, gamma=0, learning_rate=0.03, max_depth=5, n_estimators=200, reg_alpha=0, reg_lambda=1, subsample=0.9; total time=   2.2s
[CV] END colsample_bytree=0.8, gamma=0, learning_rate=0.03, max_depth=5, n_estimators=200, reg_alpha=0, reg_lambda=1, subsample=0.9; total time=   2.1s
[CV] END colsample_bytree

KeyboardInterrupt: 

In [17]:
from sklearn.model_selection import RandomizedSearchCV
import scipy.stats as stats

param_dist = {
    'n_estimators': stats.randint(200, 600),
    'max_depth': stats.randint(4, 10),
    'learning_rate': stats.uniform(0.01, 0.1),
    'subsample': stats.uniform(0.7, 0.3),
    'colsample_bytree': stats.uniform(0.7, 0.3),
    'gamma': stats.uniform(0, 2),
    'min_child_weight': stats.randint(1, 6),
    'reg_alpha': stats.uniform(0, 2),
    'reg_lambda': stats.uniform(1, 3)
}

random_search = RandomizedSearchCV(
    estimator=xgb_clf,
    param_distributions=param_dist,
    n_iter=30,
    scoring='f1_macro',
    cv=3,
    verbose=2,
    random_state=42,
    n_jobs=1
)

random_search.fit(x_train, y_train, sample_weight=sample_weights)

print("Best Params:", random_search.best_params_)
print("Best Score:", random_search.best_score_)


Fitting 3 folds for each of 30 candidates, totalling 90 fits
[CV] END colsample_bytree=0.8123620356542087, gamma=1.9014286128198323, learning_rate=0.0831993941811405, max_depth=8, min_child_weight=5, n_estimators=302, reg_alpha=0.8916655057071823, reg_lambda=1.2999247474540088, subsample=0.8377746675897602; total time=   3.9s
[CV] END colsample_bytree=0.8123620356542087, gamma=1.9014286128198323, learning_rate=0.0831993941811405, max_depth=8, min_child_weight=5, n_estimators=302, reg_alpha=0.8916655057071823, reg_lambda=1.2999247474540088, subsample=0.8377746675897602; total time=   3.3s
[CV] END colsample_bytree=0.8123620356542087, gamma=1.9014286128198323, learning_rate=0.0831993941811405, max_depth=8, min_child_weight=5, n_estimators=302, reg_alpha=0.8916655057071823, reg_lambda=1.2999247474540088, subsample=0.8377746675897602; total time=   2.9s
[CV] END colsample_bytree=0.8001125833417065, gamma=0.28573363584388156, learning_rate=0.07508884729488528, max_depth=8, min_child_weight=

In [20]:
import optuna
import numpy as np
import xgboost as xgb

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score

def objective(trial):

    params = {
        'objective': 'multi:softprob',
        'num_class': 4,
        'n_estimators': trial.suggest_int('n_estimators', 200, 600),
        'max_depth': trial.suggest_int('max_depth', 4, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1),
        'subsample': trial.suggest_float('subsample', 0.7, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.7, 1.0),
        'gamma': trial.suggest_float('gamma', 0, 2),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 6),
        'reg_alpha': trial.suggest_float('reg_alpha', 0, 2),
        'reg_lambda': trial.suggest_float('reg_lambda', 1, 3),
        'random_state': 42,
        'n_jobs': -1
    }

    model = xgb.XGBClassifier(**params)

    skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

    f1_scores = []

    for train_idx, val_idx in skf.split(x_train, y_train):

        X_tr = x_train.iloc[train_idx]
        X_val = x_train.iloc[val_idx]
        y_tr = y_train.iloc[train_idx]
        y_val = y_train.iloc[val_idx]

        # Match sample weights to fold
        sw_tr = sample_weights[train_idx]

        model.fit(X_tr, y_tr, sample_weight=sw_tr)

        y_pred = model.predict(X_val)

        f1 = f1_score(y_val, y_pred, average='macro')
        f1_scores.append(f1)

    return np.mean(f1_scores)


study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=30)

print("Best Params:", study.best_params)
print("Best Score:", study.best_value)



[I 2026-04-24 23:16:01,851] A new study created in memory with name: no-name-3d0ed598-0cdc-4420-b566-8029763d5874
[I 2026-04-24 23:16:14,174] Trial 0 finished with value: 0.43293283255222 and parameters: {'n_estimators': 541, 'max_depth': 6, 'learning_rate': 0.08382726570781286, 'subsample': 0.9005632055075415, 'colsample_bytree': 0.9826142041901045, 'gamma': 1.3674441399137442, 'min_child_weight': 2, 'reg_alpha': 1.29847801411662, 'reg_lambda': 2.103620513099294}. Best is trial 0 with value: 0.43293283255222.
[I 2026-04-24 23:16:24,349] Trial 1 finished with value: 0.4343419616865883 and parameters: {'n_estimators': 470, 'max_depth': 5, 'learning_rate': 0.09123403670861144, 'subsample': 0.8085006773680585, 'colsample_bytree': 0.9869789930110953, 'gamma': 1.3739916386529214, 'min_child_weight': 3, 'reg_alpha': 0.79691197500193, 'reg_lambda': 2.5826224289999415}. Best is trial 1 with value: 0.4343419616865883.
[I 2026-04-24 23:16:34,157] Trial 2 finished with value: 0.4285075621429542 a

Best Params: {'n_estimators': 340, 'max_depth': 10, 'learning_rate': 0.03931684970805624, 'subsample': 0.7010453350240755, 'colsample_bytree': 0.9031897997406457, 'gamma': 0.4208645825439671, 'min_child_weight': 1, 'reg_alpha': 0.05266517583919028, 'reg_lambda': 1.747709250930067}
Best Score: 0.45936419118234006


In [21]:
import xgboost as xgb

best_params = {
    'objective': 'multi:softprob',
    'num_class': 4,
    'n_estimators': 389,
    'max_depth': 9,
    'learning_rate': 0.0827,
    'subsample': 0.7814,
    'colsample_bytree': 0.8185,
    'gamma': 1.8533,
    'min_child_weight': 4,
    'reg_alpha': 0.6507,
    'reg_lambda': 2.1660,
    'random_state': 42,
    'n_jobs': -1
}

final_model = xgb.XGBClassifier(**best_params)

final_model.fit(
    x_train,
    y_train,
    sample_weight=sample_weights
)


,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'multi:softprob'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,0.8185
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

y_pred = final_model.predict(x_test)

print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))


              precision    recall  f1-score   support

           0       0.45      0.84      0.59      1014
           1       0.78      0.44      0.56      5045
           2       0.26      0.34      0.29      1325
           3       0.28      0.52      0.36      1029

    accuracy                           0.48      8413
   macro avg       0.44      0.53      0.45      8413
weighted avg       0.59      0.48      0.50      8413

[[ 852  131   15   16]
 [ 869 2224 1004  948]
 [ 127  339  450  409]
 [  36  170  292  531]]


In [23]:
y_pred_train = final_model.predict(x_train)
y_pred_test = final_model.predict(x_test)

print("=== TRAIN REPORT ===")
print(classification_report(y_train, y_pred_train))

print("=== TEST REPORT ===")
print(classification_report(y_test, y_pred_test))

print("Train Accuracy:", accuracy_score(y_train, y_pred_train))
print("Test Accuracy:", accuracy_score(y_test, y_pred_test))

print("Confusion Matrix (Test):")
print(confusion_matrix(y_test, y_pred_test))

=== TRAIN REPORT ===
              precision    recall  f1-score   support

           0       0.49      0.94      0.64      3894
           1       0.85      0.46      0.60     20407
           2       0.36      0.54      0.44      5115
           3       0.41      0.73      0.52      4235

    accuracy                           0.56     33651
   macro avg       0.53      0.67      0.55     33651
weighted avg       0.68      0.56      0.57     33651

=== TEST REPORT ===
              precision    recall  f1-score   support

           0       0.45      0.84      0.59      1014
           1       0.78      0.44      0.56      5045
           2       0.26      0.34      0.29      1325
           3       0.28      0.52      0.36      1029

    accuracy                           0.48      8413
   macro avg       0.44      0.53      0.45      8413
weighted avg       0.59      0.48      0.50      8413

Train Accuracy: 0.563549374461383
Test Accuracy: 0.48222988232497327
Confusion Matrix (Te

In [24]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import numpy as np

# 🔥 Use probabilities instead of direct predict
y_proba_train = final_model.predict_proba(x_train)
y_proba_test = final_model.predict_proba(x_test)

# Custom threshold logic (tune these!)
def custom_predict(proba):
    preds = []
    for p in proba:
        if p[3] > 0.45:      # stricter for Class 3 (reduces overprediction)
            preds.append(3)
        elif p[2] > 0.35:    # boost weak Class 2
            preds.append(2)
        else:
            preds.append(np.argmax(p))
    return np.array(preds)

y_pred_train = custom_predict(y_proba_train)
y_pred_test = custom_predict(y_proba_test)

print("=== TRAIN REPORT ===")
print(classification_report(y_train, y_pred_train))

print("=== TEST REPORT ===")
print(classification_report(y_test, y_pred_test))

print("Train Accuracy:", accuracy_score(y_train, y_pred_train))
print("Test Accuracy:", accuracy_score(y_test, y_pred_test))

print("Confusion Matrix (Test):")
print(confusion_matrix(y_test, y_pred_test))


=== TRAIN REPORT ===
              precision    recall  f1-score   support

           0       0.49      0.94      0.64      3894
           1       0.86      0.42      0.56     20407
           2       0.33      0.63      0.44      5115
           3       0.43      0.69      0.53      4235

    accuracy                           0.54     33651
   macro avg       0.53      0.67      0.54     33651
weighted avg       0.68      0.54      0.55     33651

=== TEST REPORT ===
              precision    recall  f1-score   support

           0       0.45      0.84      0.59      1014
           1       0.77      0.39      0.52      5045
           2       0.24      0.41      0.30      1325
           3       0.28      0.46      0.35      1029

    accuracy                           0.46      8413
   macro avg       0.44      0.53      0.44      8413
weighted avg       0.59      0.46      0.47      8413

Train Accuracy: 0.5428070488247007
Test Accuracy: 0.45786283133246164
Confusion Matrix (T